# Pre-train、SFT、Post-RL Training 教学实验

这个 notebook 用一个 tiny GPT 演示 LLM 训练的三个核心阶段：pre-train、SFT、post-RL/DPO。它是教学实验，不追求生成质量。重点是看清楚每个阶段的数据、loss 和训练目标。

## 0. 三阶段总览

```mermaid
flowchart LR
    A[Raw text] --> B[Pre-train: next-token prediction]
    B --> C[Base model]
    D[Instruction data] --> E[SFT: supervised fine-tuning]
    C --> E
    E --> F[Instruction model]
    G[Preference pairs] --> H[Post-RL: DPO / RLHF / GRPO]
    F --> H
    H --> I[Aligned model]
```

- **Pre-train**：用原始文本预测下一个 token，学习语言分布。
- **SFT**：用指令和标准答案训练，学习对话格式和指令跟随。
- **Post-RL**：用偏好、奖励或验证器进一步对齐模型行为。本 notebook 手写 DPO loss。

In [ ]:
import copy
import time

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from tiny_llm_training import (
    CharTokenizer,
    TinyGPT,
    build_toy_corpus,
    build_sft_examples,
    build_preference_examples,
    collect_all_texts,
    dpo_loss,
    generate,
    make_dpo_batch,
    make_sft_batch,
    sample_pretrain_batch,
    set_seed,
)

set_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. 构造 tokenizer、toy corpus 和 tiny GPT

这里用 character-level tokenizer，避免依赖外部 tokenizer 或下载模型。真实 LLM 会用 BPE/SentencePiece 等 tokenizer。

In [ ]:
tokenizer = CharTokenizer(collect_all_texts())
corpus = build_toy_corpus()
corpus_ids = tokenizer.encode(corpus, add_bos=True, add_eos=True)

max_seq_len = 96
model = TinyGPT(
    vocab_size=tokenizer.vocab_size,
    max_seq_len=max_seq_len,
    d_model=128,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print('vocab_size =', tokenizer.vocab_size)
print('corpus_tokens =', len(corpus_ids))
print('parameters =', f'{num_params:,}')

## 2. Pre-train：Next-token Prediction

Pre-train 的输入和标签基本相同，模型在位置 `t` 预测 `t+1` 的 token。

```text
input:  x0 x1 x2 ... xT
target:    x1 x2 ... xT
```

真实 pre-train 的难点通常不在这个 loss，而在数据质量、数据配比、分布式稳定性、checkpoint、吞吐和评估。

In [ ]:
pretrain_optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
pretrain_losses = []

start = time.time()
for step in range(160):
    model.train()
    input_ids, labels = sample_pretrain_batch(corpus_ids, batch_size=32, block_size=max_seq_len, device=device)
    _, loss = model(input_ids, labels=labels)
    pretrain_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    pretrain_optimizer.step()
    pretrain_losses.append(loss.item())
    if (step + 1) % 40 == 0:
        print('pretrain step', step + 1, 'loss', round(loss.item(), 4))

print('seconds =', round(time.time() - start, 2))
print(generate(model, tokenizer, 'Pre-train ', max_new_tokens=100, temperature=0.8, device=device))

## 3. SFT：只监督 Assistant Tokens

SFT 仍然是 causal LM loss，但 labels 会 mask 掉 prompt/user 部分：

```text
input:  <bos>User: 什么是 SFT?\nAssistant: SFT 是...
labels: -100 -100 -100 ...          SFT 是...
```

`-100` 是 PyTorch cross entropy 的 ignore index。这样模型不会被训练去“生成用户问题”，只会学习 assistant 应该如何回答。

In [ ]:
sft_examples = build_sft_examples()
sft_optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
sft_losses = []

for step in range(180):
    model.train()
    input_ids, labels = make_sft_batch(tokenizer, sft_examples, batch_size=16, max_seq_len=max_seq_len, device=device)
    _, loss = model(input_ids, labels=labels)
    sft_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    sft_optimizer.step()
    sft_losses.append(loss.item())
    if (step + 1) % 45 == 0:
        print('sft step', step + 1, 'loss', round(loss.item(), 4))

prompt = '<bos>User: 什么是 SFT?\nAssistant: '
print(generate(model, tokenizer, prompt, max_new_tokens=120, temperature=0.7, device=device))

SFT 结束后，把当前模型拷贝为 reference model。DPO 会冻结 reference，只训练 policy。

In [ ]:
reference_model = copy.deepcopy(model).to(device)
reference_model.eval()
for p in reference_model.parameters():
    p.requires_grad_(False)

policy_model = model

## 4. Post-RL / DPO：用偏好对优化模型

DPO 的数据是：

```text
prompt, chosen_response, rejected_response
```

核心 loss：

```text
L_DPO = -log sigmoid(beta * ((log pi_chosen - log pi_rejected)
                           - (log ref_chosen - log ref_rejected)))
```

直觉：如果 policy 相对 reference 更偏好 chosen，loss 下降。`beta` 控制偏好优化强度。

In [ ]:
preferences = build_preference_examples()
dpo_optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-4, weight_decay=0.01)
dpo_losses = []
dpo_accs = []
policy_margins = []

for step in range(120):
    policy_model.train()
    batch = make_dpo_batch(tokenizer, preferences, batch_size=8, max_seq_len=max_seq_len, device=device)
    loss, stats = dpo_loss(policy_model, reference_model, batch, beta=0.2)
    dpo_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
    dpo_optimizer.step()
    dpo_losses.append(stats['dpo_loss'])
    dpo_accs.append(stats['preference_accuracy'])
    policy_margins.append(stats['policy_margin'])
    if (step + 1) % 30 == 0:
        print('dpo step', step + 1, stats)

print(generate(policy_model, tokenizer, '<bos>User: DPO 需要什么数据?\nAssistant: ', max_new_tokens=120, temperature=0.7, device=device))

## 5. 曲线和指标

教学版主要看三类东西：

- pre-train loss：模型是否学会 toy corpus 的局部分布。
- SFT loss：assistant token 的监督是否下降。
- DPO preference accuracy / margin：policy 是否更偏好 chosen。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(pretrain_losses)
axes[0].set_title('Pre-train CE loss')
axes[0].set_xlabel('step')
axes[0].grid(True)

axes[1].plot(sft_losses)
axes[1].set_title('SFT assistant-token CE loss')
axes[1].set_xlabel('step')
axes[1].grid(True)

axes[2].plot(dpo_losses, label='DPO loss')
axes[2].plot(dpo_accs, label='preference acc')
axes[2].plot(policy_margins, label='policy margin')
axes[2].set_title('DPO metrics')
axes[2].set_xlabel('step')
axes[2].grid(True)
axes[2].legend()

plt.tight_layout()
plt.show()

## 6. 迁移到真实训练

这个 notebook 跑的是 tiny local demo。迁移到真实 LLM 时，对应关系如下：

| 阶段 | 教学代码 | 真实训练 |
|---|---|---|
| Pre-train | `TinyGPT + next-token loss` | Megatron/NeMo/DeepSpeed/FSDP |
| SFT | assistant-token masked CE | `trl.SFTTrainer` / 自写 Trainer |
| DPO | 手写 `dpo_loss` | `trl.DPOTrainer` |
| GRPO/RL | notebook 只解释概念 | `trl.GRPOTrainer` + reward function |

真实项目最容易出问题的地方通常不是公式，而是数据：去重、污染、格式、长度截断、chosen/rejected 质量、评估集泄漏和 reward hacking。